In [ ]:
%pip install seaborn

In [ ]:
#import basic libraries
import pandas as pd
import numpy as np
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier          # <- replaces DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#LOAD THE DATASET
df = pd.read_csv('phishing_site_urls.csv')


In [ ]:
#INSPECT THE DATASET
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.Label.value_counts()

In [ ]:
#TOKENIZING
from nltk.tokenize import RegexpTokenizer

In [ ]:
tokenizer = RegexpTokenizer(r'[A-Za-z]+')

In [ ]:
df.URL[0]

In [ ]:
tokenizer.tokenize(df.URL[0])

In [ ]:
df['text_tokenized']= df.URL.map(lambda t: tokenizer.tokenize(t))

In [ ]:
df.head

In [ ]:
from nltk.stem.snowball import SnowballStemmer

In [ ]:
stemmer = SnowballStemmer('english')

In [ ]:
df['text_stemmed'] = df['text_tokenized'].map(lambda l: [stemmer.stem(word) for word in l])

In [ ]:
df.head()

In [ ]:
df['text'] = df['text_stemmed'].map(lambda l: ' '.join(l))

In [ ]:
df.head()

In [ ]:
good_sites = df[df.Label == 'good']
bad_sites = df[df.Label == 'bad']

In [ ]:
good_sites.head()

In [ ]:
def plot_wordcloud( text, mask=None, max_words=400, max_front_size=120, figure_size=(24.0,16.0),
                   title = None, title_size=40, image_color=False):
    stopwords = set(STOPWORDS)
    more_stopwords = {'com','http'}
    stopwords = stopwords.union(more_stopwords)
    
    wordcloud = WordCloud(background_color='white',
                          stopwords = stopwords,
                          max_words = max_words,
                          max_font_size = max_front_size,
                          random_state = 42,
                          mask = mask)
    wordcloud.generate(text)
    
    plt.figure(figsize=figure_size)
    if image_color:
        image_colors = ImageColorGenerator(mask);
        plt.imshow(wordcloud.recolor(color_func=image_colors), interpolation= "bilinear");
        plt.title(title, fontdict={'size': title_size, 'verticalalignment': 'bottom'})
        
    else:
        plt.imshow(wordcloud);
        plt.title(title, fontdict={'size': title_size, 'color': 'green',
                                   'verticalalignment': 'bottom'})
        
        plt.axis('off')
        plt.tight_layout()

In [ ]:
all_text = ' '.join(good_sites['text'].tolist())

In [ ]:
from wordcloud import WordCloud

In [ ]:
#generate word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_text)

#display the word cloud
plt.figure(figsize=(10 , 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

In [ ]:
all_text = ' '.join(bad_sites['text'].tolist())

In [ ]:
#generate word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_text)

#display the word cloud
plt.figure(figsize=(10 , 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

In [ ]:
df.head()

In [ ]:
#convert to vector
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
cv = CountVectorizer()

In [ ]:
features= cv.fit_transform(df.text)

In [ ]:
features[:5].toarray()

In [ ]:
#splitting data into testing and training data
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(features, df.Label)

### MODEL TRAINING

### Logistic Regression

In [ ]:
#MODEL TRAINING
#logistic regression
from sklearn.linear_model import LogisticRegression

In [ ]:
l_model = LogisticRegression()

In [ ]:
l_model.fit(x_train, y_train)

In [ ]:
l_model.score(x_test, y_test)

In [ ]:
l_model.score(x_train, y_train)
#testing and training data accuracy is almost similar hence the model does not overfits; if the training data 
#accuracy would be more and testing data accuracy would be near 90 so we could have said that the model overfits.

In [ ]:
from sklearn.metrics import classification_report
print ('\nCLASSIFICATION REPORT\n')
print(classification_report(l_model.predict(x_test),y_test,
                            target_names =['Bad','Good']))

In [ ]:
#CONFUSION MATRIX
from sklearn.metrics import confusion_matrix
con_mat = pd.DataFrame(confusion_matrix(l_model.predict(x_test), y_test),
                       columns = ['Predicted:Bad', 'Predicted:Good'],
                       index = ['Actual:Bad', 'Actual:Good'])

In [ ]:
#HEATMAP OF CONFUSION MATRIX (logistic regression)
print('\nCONFUSION MATRIX')
plt.figure(figsize = (6,4))
sns.heatmap(con_mat, annot = True, fmt = 'd', cmap ="YlGnBu")

### Naive Bayes

In [ ]:
#NAIVE BAYES (takes lesser time)
from sklearn.naive_bayes import MultinomialNB 
mnb = MultinomialNB()

#training this model
mnb.fit(x_train, y_train)

In [ ]:
mnb.score(x_test, y_test)

In [ ]:
mnb.score(x_train, y_train)

In [ ]:
from sklearn.metrics import classification_report
print ('\nCLASSIFICATION REPORT\n')
print(classification_report(mnb.predict(x_test),y_test,
                            target_names =['Bad','Good']))

In [ ]:
#CONFUSION MATRIX
from sklearn.metrics import confusion_matrix
con_mat = pd.DataFrame(confusion_matrix(mnb.predict(x_test), y_test),
                       columns = ['Predicted:Bad', 'Predicted:Good'],
                       index = ['Actual:Bad', 'Actual:Good'])

In [ ]:
#HEATMAP OF CONFUSION MATRIX (naive bayes)
print('\nCONFUSION MATRIX')
plt.figure(figsize = (6,4))
sns.heatmap(con_mat, annot = True, fmt = 'd', cmap ="YlGnBu")

RANDOM FOREST

In [ ]:
# ---- RANDOM FOREST ----
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Train the model
rf_model.fit(x_train, y_train)


In [ ]:

# Test accuracy
print('Test Accuracy :', rf_model.score(x_test, y_test))

In [ ]:
# Train accuracy
print('Train Accuracy:', rf_model.score(x_train, y_train))

In [ ]:

print('\nCLASSIFICATION REPORT\n')
print(classification_report(rf_model.predict(x_test), y_test,
                             target_names=['Bad', 'Good']))



In [ ]:

# CONFUSION MATRIX
rf_con_mat = pd.DataFrame(confusion_matrix(rf_model.predict(x_test), y_test),
                           columns=['Predicted:Bad', 'Predicted:Good'],
                           index=['Actual:Bad', 'Actual:Good'])



In [ ]:

# HEATMAP OF CONFUSION MATRIX (Random Forest)
print('\nCONFUSION MATRIX')
plt.figure(figsize=(6, 4))
sns.heatmap(rf_con_mat, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Random Forest - Confusion Matrix')
plt.show()

### SAVE MODEL

In [ ]:
import pickle
pickle.dump(l_model, open('phishing.pkl', 'wb'))

In [ ]:
pickle.dump(mnb, open('phishing_mnb.pkl', 'wb'))

In [ ]:

pickle.dump(rf_model, open('phishing_rf.pkl', 'wb'))

In [ ]:
pickle.dump(cv, open('vectorizer.pkl', 'wb'))

In [ ]:
print('All models saved successfully!')